In [ ]:
import matplotlib.pyplot as plt
import skyproj
import numpy as np
from streamobs.surveys import Survey
import os

In [ ]:
fig_folder = "./outputs"
os.makedirs(fig_folder, exist_ok=True)

# DR1

In [ ]:
des = Survey.load("des", release="yr6")
euclid_dr1 = Survey.load("euclid", release="dr1")

# Copies: the surveys are cached, their coverage must not be modified
footprint_des = des.coverage.copy()
footprint_euclid_dr1 = euclid_dr1.coverage.copy()
footprint_des[footprint_des < 1] = np.nan
footprint_euclid_dr1[footprint_euclid_dr1 < 1] = np.nan
footprint_des_euclid_dr1= footprint_des * footprint_euclid_dr1


In [ ]:
import healpy as hp
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

coord = "galactic"  # "equatorial" (ra, dec) or "galactic" (l, b: Galactic plane horizontal, in the middle)
coord = "equatorial"

def to_galactic(hpx_map):
    """Equatorial HEALPix map resampled in Galactic coordinates (nearest pixel, keeps the 1/NaN values)."""
    nside = hp.get_nside(hpx_map)
    theta, phi = hp.pix2ang(nside, np.arange(hp.nside2npix(nside)))
    theta_eq, phi_eq = hp.Rotator(coord=["G", "C"])(theta, phi)
    return hpx_map[hp.ang2pix(nside, theta_eq, phi_eq)]


# Drawn in this order, each layer above the previous one
layers = [
    ("Euclid DR1", footprint_euclid_dr1, "#0072B2"),
    ("DES Y6", footprint_des, "#E69F00"),
    ("Overlap", footprint_des_euclid_dr1, "#009E73"),
]
pixel_area = hp.nside2pixarea(hp.get_nside(footprint_des), degrees=True)

fig, ax = plt.subplots(figsize=(9, 5))
# galactic=True only changes the labels: the maps are converted below
proj = skyproj.McBrydeSkyproj(ax=ax, galactic=(coord == "galactic"))

legend_handles = []
for label, footprint, color in layers:
    area = np.sum(np.isfinite(footprint)) * pixel_area  # on the original map
    if coord == "galactic":
        footprint = to_galactic(footprint)
    # footprint: 1 inside, NaN outside; zoom=False keeps the full sky for every layer
    proj.draw_hpxmap(footprint, nest=False, zoom=False, cmap=ListedColormap([color]), vmin=0, vmax=1)
    legend_handles.append(Patch(facecolor=color, label=f"{label} ({area:,.0f} deg$^2$)"))
proj.draw_milky_way()

xlabel, ylabel = {
    "equatorial": ("Right Ascension", "Declination"),
    "galactic": ("Galactic Longitude", "Galactic Latitude"),
}[coord]
proj.ax.set_xlabel(xlabel, fontsize=10)
proj.ax.set_ylabel(ylabel, fontsize=10)
fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.95), ncol=3, frameon=False)

In [ ]:
fig.savefig(os.path.join(fig_folder, f"DES_Euclid_footprint_{coord}.pdf"), dpi=300, bbox_inches="tight")

# Q1

In [ ]:
euclid_q1 = Survey.load("euclid", release="q1")

# Copies: the surveys are cached, their coverage must not be modified
footprint_euclid_q1 = euclid_q1.coverage.copy()
footprint_euclid_q1[footprint_euclid_q1 < 1] = np.nan
footprint_des_euclid_q1= footprint_des * footprint_euclid_q1

In [ ]:
import healpy as hp
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

coord = "galactic"  # "equatorial" (ra, dec) or "galactic" (l, b: Galactic plane horizontal, in the middle)
coord = "equatorial"

def to_galactic(hpx_map):
    """Equatorial HEALPix map resampled in Galactic coordinates (nearest pixel, keeps the 1/NaN values)."""
    nside = hp.get_nside(hpx_map)
    theta, phi = hp.pix2ang(nside, np.arange(hp.nside2npix(nside)))
    theta_eq, phi_eq = hp.Rotator(coord=["G", "C"])(theta, phi)
    return hpx_map[hp.ang2pix(nside, theta_eq, phi_eq)]


# Drawn in this order, each layer above the previous one
layers = [
    ("Euclid Q1", footprint_euclid_q1, "#0072B2"),      
    ("DES Y6", footprint_des, "#E69F00"),
    ("Overlap", footprint_des_euclid_q1, "#009E73"),
]

pixel_area = hp.nside2pixarea(hp.get_nside(footprint_des), degrees=True)

fig, ax = plt.subplots(figsize=(9, 5))
# galactic=True only changes the labels: the maps are converted below
proj = skyproj.McBrydeSkyproj(ax=ax, galactic=(coord == "galactic"))

legend_handles = []
for label, footprint, color in layers:
    area = np.sum(np.isfinite(footprint)) * pixel_area  # on the original map
    if coord == "galactic":
        footprint = to_galactic(footprint)
    # footprint: 1 inside, NaN outside; zoom=False keeps the full sky for every layer
    proj.draw_hpxmap(footprint, nest=False, zoom=False, cmap=ListedColormap([color]), vmin=0, vmax=1)
    legend_handles.append(Patch(facecolor=color, label=f"{label} ({area:,.0f} deg$^2$)"))
proj.draw_milky_way()

xlabel, ylabel = {
    "equatorial": ("Right Ascension", "Declination"),
    "galactic": ("Galactic Longitude", "Galactic Latitude"),
}[coord]
proj.ax.set_xlabel(xlabel, fontsize=10)
proj.ax.set_ylabel(ylabel, fontsize=10)
fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.95), ncol=3, frameon=False)